In [ ]:
!pip install -q transformers peft trl datasets accelerate

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
data = [
    # --- Python Errors ---
    {"instruction": "Error: NameError: name 'x' is not defined", "output": "What broke: You used a variable called 'x' but never created it before using it.\nWhy it broke: Python reads code top to bottom. If you try to use a variable before assigning a value to it, Python has no idea what it is.\nHow to fix: Make sure you write x = something before the line where you use x.\nExample fix:\nx = 10\nprint(x)"},
    {"instruction": "Error: IndentationError: expected an indented block", "output": "What broke: Python expected your code to be indented (moved to the right with spaces) but found nothing.\nWhy it broke: After lines ending with a colon like if, for, def, or class, Python requires at least one indented line inside.\nHow to fix: Add 4 spaces before the code inside your if, for, or function block.\nExample fix:\nif True:\n    print('hello')"},
    {"instruction": "Error: TypeError: unsupported operand type(s) for +: 'int' and 'str'", "output": "What broke: You tried to add a number and a piece of text together which Python cannot do.\nWhy it broke: Python treats numbers and text as completely different types. You cannot mix them in math operations without converting.\nHow to fix: Convert the number to a string using str() or convert the string to a number using int().\nExample fix:\nage = 25\nprint('I am ' + str(age) + ' years old')"},
    {"instruction": "Error: IndexError: list index out of range", "output": "What broke: You tried to access a position in a list that does not exist.\nWhy it broke: Lists in Python start at index 0. If your list has 3 items, the valid indexes are 0, 1, and 2. Trying to access index 3 or higher crashes the program.\nHow to fix: Check the length of your list using len() before accessing items, or make sure your index is within range.\nExample fix:\nmy_list = [1, 2, 3]\nprint(my_list[0])  # valid, prints 1"},
    {"instruction": "Error: KeyError: 'name'", "output": "What broke: You tried to access a key called 'name' in a dictionary but that key does not exist.\nWhy it broke: Dictionaries only contain the keys you explicitly added. Asking for a key that was never added raises a KeyError.\nHow to fix: Use the get() method which returns None instead of crashing, or check if the key exists first using 'in'.\nExample fix:\nperson = {'age': 25}\nname = person.get('name', 'Unknown')"},
    {"instruction": "Error: AttributeError: 'NoneType' object has no attribute 'split'", "output": "What broke: You called .split() on a variable that contains None instead of a string.\nWhy it broke: A function returned None meaning it found nothing or failed silently. You then tried to use that None value as if it were a string.\nHow to fix: Check if your variable is None before calling methods on it.\nExample fix:\nresult = get_name()\nif result is not None:\n    words = result.split()"},
    {"instruction": "Error: ZeroDivisionError: division by zero", "output": "What broke: Your code tried to divide a number by zero.\nWhy it broke: Division by zero is mathematically undefined and Python raises an error instead of producing infinity or garbage values.\nHow to fix: Always check that your divisor is not zero before dividing.\nExample fix:\nif denominator != 0:\n    result = numerator / denominator"},
    {"instruction": "Error: FileNotFoundError: [Errno 2] No such file or directory: 'data.txt'", "output": "What broke: Python tried to open a file called data.txt but could not find it.\nWhy it broke: Either the file does not exist, the filename is spelled wrong, or the file is in a different folder than where your script is running.\nHow to fix: Check the filename spelling, make sure the file exists, and check you are looking in the right folder.\nExample fix:\nimport os\nprint(os.getcwd())  # shows your current folder\nwith open('data.txt', 'r') as f:\n    content = f.read()"},
    {"instruction": "Error: ImportError: No module named 'pandas'", "output": "What broke: Python cannot find a library called pandas on your computer.\nWhy it broke: The pandas library is not installed in your Python environment. It does not come with Python by default.\nHow to fix: Install it using pip in your terminal before running your script.\nExample fix:\npip install pandas\nThen in your code:\nimport pandas as pd"},
    {"instruction": "Error: RecursionError: maximum recursion depth exceeded", "output": "What broke: A function kept calling itself too many times and Python hit its safety limit.\nWhy it broke: You have a recursive function that either has no base case to stop, or the base case is never reached causing infinite self-calling.\nHow to fix: Make sure your recursive function has a clear stopping condition that will definitely be reached.\nExample fix:\ndef countdown(n):\n    if n <= 0:  # base case - stops recursion\n        return\n    countdown(n - 1)"},
    {"instruction": "Error: ValueError: invalid literal for int() with base 10: 'hello'", "output": "What broke: You tried to convert the text 'hello' into an integer but 'hello' is not a number.\nWhy it broke: The int() function can only convert strings that contain actual numbers like '42' or '100'. It cannot convert words or mixed text.\nHow to fix: Validate that the string is a number before converting, or use try/except to handle the error gracefully.\nExample fix:\nuser_input = input('Enter a number: ')\nif user_input.isdigit():\n    number = int(user_input)"},
    {"instruction": "Error: TypeError: 'int' object is not subscriptable", "output": "What broke: You tried to use square brackets on a number like it was a list or string.\nWhy it broke: Square brackets like [0] are used to access items in lists, strings, or dictionaries. Numbers do not support this operation.\nHow to fix: Check your variable type. You probably meant to work with a list or string but accidentally stored a number instead.\nExample fix:\nmy_list = [1, 2, 3]\nprint(my_list[0])  # correct - accessing a list"},
    {"instruction": "Error: SyntaxError: invalid syntax", "output": "What broke: Python could not understand your code because of a writing mistake.\nWhy it broke: This happens due to missing colons, unmatched brackets, typos in keywords, or incorrect punctuation that makes Python unable to parse your code.\nHow to fix: Look at the line number in the error. Common causes are missing colon after if/for/def, unclosed parentheses, or misspelled keywords.\nExample fix:\nif x > 5:  # don't forget the colon\n    print('big')"},
    {"instruction": "Error: TypeError: takes 1 positional argument but 2 were given", "output": "What broke: You called a function with more arguments than it was designed to accept.\nWhy it broke: The function definition only declares one parameter but you passed two values when calling it.\nHow to fix: Either add more parameters to the function definition or pass fewer arguments when calling it.\nExample fix:\ndef greet(name):  # accepts 1 argument\n    print('Hello', name)\ngreet('Alice')  # correct - passing 1 argument"},
    {"instruction": "Error: MemoryError", "output": "What broke: Your program ran out of RAM while trying to process data.\nWhy it broke: You tried to load or create something extremely large in memory, like reading a huge file all at once or creating a massive list.\nHow to fix: Process data in smaller chunks instead of loading everything at once.\nExample fix:\nwith open('big_file.txt') as f:\n    for line in f:  # reads one line at a time\n        process(line)"},
    {"instruction": "Error: StopIteration", "output": "What broke: You called next() on an iterator that has already run out of items.\nWhy it broke: Iterators can only be consumed once. Once you reach the end, calling next() again raises StopIteration.\nHow to fix: Use a for loop which handles StopIteration automatically, or check before calling next().\nExample fix:\nfor item in my_iterator:  # safe - handles StopIteration automatically\n    print(item)"},
    {"instruction": "Error: OverflowError: math range error", "output": "What broke: A math calculation produced a number too large for Python to handle.\nWhy it broke: Certain math operations like exponentials can grow astronomically fast and exceed the maximum float value Python can represent.\nHow to fix: Use Python's built-in integers for very large numbers which have unlimited precision, or restructure your calculation.\nExample fix:\nimport math\nresult = math.factorial(100)  # Python handles big integers natively"},
    {"instruction": "Error: UnicodeDecodeError: 'utf-8' codec can't decode byte", "output": "What broke: Python tried to read a file as UTF-8 text but found bytes that are not valid UTF-8.\nWhy it broke: The file was saved in a different encoding like latin-1 or windows-1252 but Python assumed UTF-8.\nHow to fix: Specify the correct encoding when opening the file, or use errors='ignore' to skip problematic characters.\nExample fix:\nwith open('file.txt', encoding='latin-1') as f:\n    content = f.read()"},
    {"instruction": "Error: PermissionError: [Errno 13] Permission denied", "output": "What broke: Python tried to read or write a file but the operating system blocked it due to insufficient permissions.\nWhy it broke: The file or folder is protected, read-only, or owned by a different user or process.\nHow to fix: Check file permissions, run your script with appropriate privileges, or choose a different folder you have write access to.\nExample fix:\nimport tempfile\nwith tempfile.NamedTemporaryFile() as f:  # uses temp folder where you always have permission\n    f.write(b'hello')"},
    {"instruction": "Error: ConnectionError: Failed to establish a new connection", "output": "What broke: Your code tried to connect to the internet or a server but the connection failed.\nWhy it broke: Either there is no internet connection, the server address is wrong, the server is down, or a firewall is blocking the connection.\nHow to fix: Check your internet connection, verify the URL is correct, and handle connection errors with try/except.\nExample fix:\nimport requests\ntry:\n    response = requests.get('https://api.example.com')\nexcept requests.ConnectionError:\n    print('No connection available')"},

    # --- JavaScript Errors ---
    {"instruction": "Error: TypeError: Cannot read properties of undefined (reading 'map')", "output": "What broke: You called .map() on a variable that is undefined instead of an array.\nWhy it broke: The variable was never assigned a value, or a function returned undefined instead of the array you expected.\nHow to fix: Add a check to make sure the variable is an array before calling .map() on it.\nExample fix:\nconst items = data?.items || [];\nitems.map(item => console.log(item));"},
    {"instruction": "Error: ReferenceError: variable is not defined", "output": "What broke: You used a variable that has never been declared anywhere in scope.\nWhy it broke: JavaScript cannot find any let, const, or var declaration for this variable name in the current or parent scopes.\nHow to fix: Declare the variable before using it with let, const, or var.\nExample fix:\nlet count = 0;\nconsole.log(count);"},
    {"instruction": "Error: SyntaxError: Unexpected token '}'", "output": "What broke: JavaScript found a closing curly brace where it did not expect one.\nWhy it broke: There is a mismatched bracket somewhere in your code — either an extra closing brace or a missing opening brace.\nHow to fix: Count your opening and closing curly braces carefully. Most code editors highlight matching brackets.\nExample fix:\nfunction greet() {\n    console.log('hello');\n}  // every opening { needs a closing }"},
    {"instruction": "Error: TypeError: X is not a function", "output": "What broke: You tried to call something as a function but it is not a function.\nWhy it broke: Either the variable holds a string, number, or object instead of a function, or you misspelled the function name.\nHow to fix: Check what type the variable actually holds using console.log(typeof X) before calling it.\nExample fix:\nconsole.log(typeof myVar);  // check if it's actually a function\nif (typeof myVar === 'function') {\n    myVar();\n}"},
    {"instruction": "Error: Uncaught RangeError: Maximum call stack size exceeded", "output": "What broke: A function kept calling itself infinitely until JavaScript ran out of call stack space.\nWhy it broke: You have infinite recursion — a function that calls itself with no stopping condition, or the stopping condition is never reached.\nHow to fix: Add a base case that stops the recursion, or rewrite the logic using a loop instead.\nExample fix:\nfunction countdown(n) {\n    if (n <= 0) return;  // base case stops recursion\n    countdown(n - 1);\n}"},
    {"instruction": "Error: TypeError: Cannot set properties of null (setting 'innerHTML')", "output": "What broke: You tried to set innerHTML on an element but the element is null meaning it was not found in the page.\nWhy it broke: document.getElementById() returns null when no element with that ID exists. This usually means a typo in the ID name or the script runs before the HTML loads.\nHow to fix: Check the element ID spelling and make sure your script runs after the HTML is loaded.\nExample fix:\ndocument.addEventListener('DOMContentLoaded', () => {\n    const el = document.getElementById('myDiv');\n    if (el) el.innerHTML = 'Hello';\n});"},
    {"instruction": "Error: SyntaxError: Unexpected end of input", "output": "What broke: JavaScript reached the end of your file but expected more code, usually a closing bracket or brace.\nWhy it broke: You have unclosed parentheses, brackets, or curly braces somewhere in your code.\nHow to fix: Check for missing closing brackets at the end of functions, objects, or arrays.\nExample fix:\nconst obj = {\n    name: 'Alice'\n};  // don't forget closing }"},
    {"instruction": "Error: TypeError: Cannot read properties of null (reading 'addEventListener')", "output": "What broke: You tried to add an event listener to an element that does not exist on the page.\nWhy it broke: The element was not found, either because the ID is wrong, the element does not exist yet, or the script runs before the HTML is rendered.\nHow to fix: Wrap your code in DOMContentLoaded to ensure HTML is ready, and verify the element ID is correct.\nExample fix:\ndocument.addEventListener('DOMContentLoaded', () => {\n    document.getElementById('btn').addEventListener('click', handleClick);\n});"},
    {"instruction": "Error: Uncaught TypeError: fetch is not a function", "output": "What broke: The fetch API is not available in the environment where your code is running.\nWhy it broke: fetch is a browser API not available in older Node.js versions or certain restricted environments.\nHow to fix: In Node.js install the node-fetch package, or use the built-in fetch available in Node 18 and above.\nExample fix:\n// Node.js below 18\nconst fetch = require('node-fetch');\nfetch('https://api.example.com');"},
    {"instruction": "Error: CORS policy: No 'Access-Control-Allow-Origin' header", "output": "What broke: Your browser blocked a network request because the server did not give permission for your website to access it.\nWhy it broke: Browsers enforce CORS security policy which prevents websites from making requests to different domains unless the server explicitly allows it.\nHow to fix: This needs to be fixed on the server side by adding CORS headers. For development you can use a proxy.\nExample fix:\n// In your Express server\napp.use((req, res, next) => {\n    res.header('Access-Control-Allow-Origin', '*');\n    next();\n});"},

    # --- SQL Errors ---
    {"instruction": "Error: SQL syntax error near 'WHERE'", "output": "What broke: Your SQL query has a syntax mistake near the WHERE keyword.\nWhy it broke: Common causes are a missing comma in the SELECT list, using a reserved word as a column name, or placing WHERE in the wrong position.\nHow to fix: Check the clause just before WHERE. Make sure all selected columns are separated by commas and no keywords are used as identifiers without backticks.\nExample fix:\nSELECT name, age\nFROM users\nWHERE age > 18;"},
    {"instruction": "Error: ORA-00942: table or view does not exist", "output": "What broke: You tried to query a table that the database cannot find.\nWhy it broke: Either the table name is spelled wrong, it exists in a different schema, or you do not have permission to access it.\nHow to fix: Check the exact table name using a database browser, verify you are connected to the correct database, and check your permissions.\nExample fix:\nSELECT * FROM information_schema.tables\nWHERE table_name = 'users';  -- verify the table exists"},
    {"instruction": "Error: ERROR 1062: Duplicate entry for key 'PRIMARY'", "output": "What broke: You tried to insert a row with a primary key value that already exists in the table.\nWhy it broke: Primary keys must be unique. You are trying to add a duplicate ID or unique value that conflicts with an existing record.\nHow to fix: Use INSERT IGNORE to skip duplicates, or ON DUPLICATE KEY UPDATE to update instead of inserting.\nExample fix:\nINSERT INTO users (id, name)\nVALUES (1, 'Alice')\nON DUPLICATE KEY UPDATE name = 'Alice';"},
    {"instruction": "Error: column 'user_id' does not exist", "output": "What broke: Your SQL query references a column called user_id but it does not exist in that table.\nWhy it broke: Either the column name is misspelled, the column is in a different table, or it was never created.\nHow to fix: Check the actual column names using DESCRIBE tablename or look at the schema.\nExample fix:\nDESCRIBE users;  -- shows all column names in the users table\nSELECT id, name FROM users;  -- use the correct column names"},
    {"instruction": "Error: ERROR 1452: Cannot add or update a child row: a foreign key constraint fails", "output": "What broke: You tried to insert or update a row but the foreign key value you provided does not exist in the parent table.\nWhy it broke: Foreign key constraints ensure data integrity by requiring that values in one table must exist in the referenced table first.\nHow to fix: First insert the record in the parent table, then insert in the child table.\nExample fix:\n-- First insert the parent\nINSERT INTO departments (id, name) VALUES (1, 'Engineering');\n-- Then insert the child\nINSERT INTO employees (id, dept_id) VALUES (100, 1);"},

    # --- Java Errors ---
    {"instruction": "Error: NullPointerException at line 42", "output": "What broke: Your code tried to use an object that is null as if it had a value.\nWhy it broke: A variable was declared but never assigned an actual object, or a method returned null and you used the result without checking.\nHow to fix: Always check if an object is null before calling methods on it.\nExample fix:\nString name = getName();\nif (name != null) {\n    System.out.println(name.length());\n}"},
    {"instruction": "Error: ArrayIndexOutOfBoundsException: Index 5 out of bounds for length 3", "output": "What broke: You tried to access position 5 in an array that only has 3 elements.\nWhy it broke: Java arrays are zero-indexed so a 3-element array has valid indexes 0, 1, and 2 only.\nHow to fix: Always use array.length to check the size before accessing elements.\nExample fix:\nint[] arr = {10, 20, 30};\nfor (int i = 0; i < arr.length; i++) {\n    System.out.println(arr[i]);\n}"},
    {"instruction": "Error: ClassCastException: String cannot be cast to Integer", "output": "What broke: You tried to treat a String object as if it were an Integer object.\nWhy it broke: Java is strongly typed and you cannot directly cast incompatible types. The object in memory is a String but you told Java to treat it as Integer.\nHow to fix: Use Integer.parseInt() to convert a String to an int properly.\nExample fix:\nString numStr = '42';\nint num = Integer.parseInt(numStr);  // correct conversion"},
    {"instruction": "Error: StackOverflowError", "output": "What broke: A method kept calling itself or other methods in an endless chain until Java ran out of stack space.\nWhy it broke: You have infinite recursion with no base case, or two methods keep calling each other in an endless loop.\nHow to fix: Find the recursive call and add a proper base case that stops the recursion.\nExample fix:\npublic int factorial(int n) {\n    if (n <= 1) return 1;  // base case\n    return n * factorial(n - 1);\n}"},
    {"instruction": "Error: NumberFormatException: For input string 'abc'", "output": "What broke: You tried to convert the text 'abc' into a number but it is not a valid number.\nWhy it broke: Integer.parseInt() and similar methods can only handle strings that contain actual numeric digits.\nHow to fix: Validate the string before converting or wrap in try-catch.\nExample fix:\ntry {\n    int num = Integer.parseInt(input);\n} catch (NumberFormatException e) {\n    System.out.println('Please enter a valid number');\n}"},

    # --- React / Frontend Errors ---
    {"instruction": "Error: Each child in a list should have a unique 'key' prop", "output": "What broke: You rendered a list of elements in React without providing a unique key prop for each item.\nWhy it broke: React needs a unique key on each list item to efficiently track which items changed, were added, or were removed during re-renders.\nHow to fix: Add a key prop to each item in your list, using a unique identifier like an ID.\nExample fix:\nitems.map(item => (\n    <div key={item.id}>{item.name}</div>\n))"},
    {"instruction": "Error: React Hook useState is called conditionally", "output": "What broke: You called a React hook like useState inside an if statement or loop.\nWhy it broke: React requires hooks to always be called in the same order on every render. Calling them conditionally breaks this rule and causes unpredictable behavior.\nHow to fix: Move all hook calls to the top level of your component, outside of any conditions or loops.\nExample fix:\nfunction MyComponent() {\n    const [count, setCount] = useState(0);  // always at top level\n    if (someCondition) {\n        // use count here, not useState\n    }\n}"},
    {"instruction": "Error: Cannot update a component while rendering a different component", "output": "What broke: You called a state setter function directly during the render phase of another component.\nWhy it broke: Calling setState during render causes an infinite update loop because setting state triggers a re-render which calls setState again.\nHow to fix: Move state updates into event handlers or useEffect hooks, never call them directly during render.\nExample fix:\nuseEffect(() => {\n    setCount(newValue);  // safe - runs after render\n}, [dependency]);"},
    {"instruction": "Error: Too many re-renders. React limits the number of renders", "output": "What broke: Your component is stuck in an infinite loop of re-renders.\nWhy it broke: You are calling a state setter function directly in the component body without any condition, causing render to trigger setState to trigger render endlessly.\nHow to fix: Make sure state updates only happen in event handlers or inside useEffect with proper dependencies.\nExample fix:\n// Wrong\nsetCount(count + 1);  // called during render\n// Correct\n<button onClick={() => setCount(count + 1)}>Click</button>"},
    {"instruction": "Error: useEffect has a missing dependency warning", "output": "What broke: Your useEffect uses a variable that is not listed in the dependency array.\nWhy it broke: React cannot guarantee that the effect has access to the latest version of the variable if it is not declared as a dependency.\nHow to fix: Add the missing variable to the dependency array, or use useCallback if it is a function.\nExample fix:\nuseEffect(() => {\n    fetchData(userId);  // userId is used\n}, [userId]);  // so userId must be in dependencies"},

    # --- Git Errors ---
    {"instruction": "Error: git merge conflict in file main.py", "output": "What broke: Two different versions of the same file were changed and Git cannot automatically decide which changes to keep.\nWhy it broke: You and someone else (or you on different branches) both edited the same lines in the same file and Git needs a human to decide which version is correct.\nHow to fix: Open the conflicted file, look for the conflict markers, decide which code to keep, remove the markers, then commit.\nExample fix:\n<<<<<<< HEAD\nyour changes here\n=======\ntheir changes here\n>>>>>>> branch-name\n# Delete the markers and keep the code you want"},
    {"instruction": "Error: git push rejected: Updates were rejected because remote contains work", "output": "What broke: Git refused to push your changes because the remote branch has commits that you do not have locally.\nWhy it broke: Someone else pushed changes to the same branch after your last pull, so your local branch is behind.\nHow to fix: Pull the latest changes first, resolve any conflicts, then push again.\nExample fix:\ngit pull origin main\n# resolve any conflicts\ngit push origin main"},
    {"instruction": "Error: detached HEAD state", "output": "What broke: Your Git repository is in a special state where you are not on any branch.\nWhy it broke: You checked out a specific commit hash instead of a branch name, putting Git into detached HEAD mode.\nHow to fix: Create a new branch from your current position to save your work, or switch back to an existing branch.\nExample fix:\ngit checkout -b my-new-branch  # save current state as new branch\n# or\ngit checkout main  # go back to main branch"},

    # --- Python Libraries ---
    {"instruction": "Error: ModuleNotFoundError: No module named 'cv2'", "output": "What broke: Python cannot find the OpenCV library which is imported as cv2.\nWhy it broke: OpenCV is not installed in your current Python environment.\nHow to fix: Install it using pip with the correct package name, which is opencv-python not cv2.\nExample fix:\npip install opencv-python\nThen in your code:\nimport cv2"},
    {"instruction": "Error: pandas DataFrame KeyError: 'column_name'", "output": "What broke: You tried to access a column that does not exist in your pandas DataFrame.\nWhy it broke: Either the column name is misspelled, has extra spaces, or does not exist in the data.\nHow to fix: Print df.columns to see all available column names and check for typos or spaces.\nExample fix:\nprint(df.columns.tolist())  # see all column names\nprint(df['actual_column_name'])  # use the exact name"},
    {"instruction": "Error: sklearn ValueError: Input contains NaN", "output": "What broke: A scikit-learn model received data containing NaN (missing values) which it cannot process.\nWhy it broke: Scikit-learn models require clean numeric data with no missing values by default.\nHow to fix: Remove or fill missing values before passing data to the model.\nExample fix:\nimport pandas as pd\nfrom sklearn.impute import SimpleImputer\nimputer = SimpleImputer(strategy='mean')\nX_clean = imputer.fit_transform(X)"},
    {"instruction": "Error: TensorFlow ResourceExhaustedError: OOM when allocating tensor", "output": "What broke: Your GPU or CPU ran out of memory while trying to allocate a tensor for your model.\nWhy it broke: Your batch size is too large, the model is too big, or there are memory leaks from previous operations not being cleared.\nHow to fix: Reduce batch size, clear GPU memory between runs, or use gradient checkpointing.\nExample fix:\n# Reduce batch size\nmodel.fit(X, y, batch_size=16)  # try smaller values like 8 or 4\n# Clear memory\nimport tensorflow as tf\ntf.keras.backend.clear_session()"},
    {"instruction": "Error: requests.exceptions.JSONDecodeError: Expecting value", "output": "What broke: You called .json() on an API response but the response body is not valid JSON.\nWhy it broke: The server returned an error page, empty response, or HTML instead of the JSON you expected.\nHow to fix: Check the response status code and print the raw text before trying to parse JSON.\nExample fix:\nresponse = requests.get(url)\nprint(response.status_code)\nprint(response.text)  # see what was actually returned\nif response.status_code == 200:\n    data = response.json()"},
    {"instruction": "Error: SQLAlchemy OperationalError: no such table", "output": "What broke: SQLAlchemy tried to query a database table that does not exist.\nWhy it broke: Either the table was never created, migrations have not been run, or you are connected to a different database than expected.\nHow to fix: Run your database migrations or create the table first.\nExample fix:\n# Create tables from your models\nBase.metadata.create_all(engine)\n# Or run migrations\n# flask db upgrade"},
    {"instruction": "Error: PIL UnidentifiedImageError: cannot identify image file", "output": "What broke: Pillow tried to open a file as an image but could not recognize its format.\nWhy it broke: The file is either corrupted, not actually an image, has the wrong extension, or is an unsupported format.\nHow to fix: Verify the file is a valid image by opening it manually, check the file path, and handle the error gracefully.\nExample fix:\nfrom PIL import Image\ntry:\n    img = Image.open('photo.jpg')\nexcept Exception as e:\n    print(f'Could not open image: {e}')"},
    {"instruction": "Error: OSError: [Errno 28] No space left on device", "output": "What broke: Your program tried to write a file but the disk is completely full.\nWhy it broke: The storage drive has no free space remaining, which can happen when training ML models, downloading datasets, or writing large output files.\nHow to fix: Free up disk space by deleting unnecessary files, or write to a different drive with available space.\nExample fix:\nimport shutil\ntotal, used, free = shutil.disk_usage('/')\nprint(f'Free space: {free // (2**30)} GB')  # check space before writing"},
    {"instruction": "Error: json.decoder.JSONDecodeError: Expecting ',' delimiter", "output": "What broke: Python tried to parse a JSON string but found invalid syntax, specifically a missing comma.\nWhy it broke: The JSON data is malformed. This often happens when manually writing JSON, using single quotes instead of double quotes, or having a trailing comma.\nHow to fix: Validate your JSON using a JSON validator tool and make sure all strings use double quotes.\nExample fix:\nimport json\n# Invalid JSON\nbad = '{name: Alice}'  # missing quotes around key\n# Valid JSON\ngood = '{\"name\": \"Alice\"}'\ndata = json.loads(good)"},
    {"instruction": "Error: TimeoutError: timed out after 30 seconds", "output": "What broke: An operation took longer than the allowed time limit and was forcibly stopped.\nWhy it broke: A network request, database query, or computation took too long, either due to a slow server, infinite loop, or large dataset.\nHow to fix: Increase the timeout limit, optimize the slow operation, or add proper timeout handling.\nExample fix:\nimport requests\ntry:\n    response = requests.get(url, timeout=60)  # increase timeout\nexcept requests.Timeout:\n    print('Request timed out, try again later')"},
    {"instruction": "Error: AssertionError", "output": "What broke: An assert statement in your code found that a condition it expected to be true was actually false.\nWhy it broke: Assert statements are used to verify assumptions about data or program state. When the assumption is wrong the program crashes immediately.\nHow to fix: Find the assert statement that failed and investigate why the condition is false at that point in execution.\nExample fix:\nvalue = calculate_result()\nprint(value)  # debug: see what value actually is\nassert value > 0, f'Expected positive but got {value}'"},
    {"instruction": "Error: RuntimeError: CUDA out of memory", "output": "What broke: Your GPU ran out of memory while running a PyTorch operation.\nWhy it broke: The model, batch size, or data is too large to fit in your GPU's VRAM at the same time.\nHow to fix: Reduce batch size, clear GPU cache between operations, or use gradient checkpointing.\nExample fix:\nimport torch\ntorch.cuda.empty_cache()  # clear unused memory\n# Reduce batch size in training\nfor batch in dataloader:  # use smaller batch_size in DataLoader\n    output = model(batch)"},
    {"instruction": "Error: ModuleNotFoundError: No module named '__main__.models'", "output": "What broke: Python cannot find a module using relative import when running the file directly.\nWhy it broke: Relative imports only work when a file is part of a package and run as a module, not when run directly with python script.py.\nHow to fix: Use absolute imports instead of relative imports, or run the script as a module.\nExample fix:\n# Instead of relative import\n# from .models import User  \n# Use absolute import\nfrom myapp.models import User\n# Or run as module\n# python -m myapp.main"},
    {"instruction": "Error: AttributeError: module 'numpy' has no attribute 'float'", "output": "What broke: You used np.float which was removed in NumPy version 1.24.\nWhy it broke: numpy deprecated and removed several type aliases like np.float, np.int, np.bool that used to exist as shortcuts.\nHow to fix: Replace np.float with the built-in Python float or use np.float64 for the NumPy specific type.\nExample fix:\n# Old broken code\narr = np.array([1.0], dtype=np.float)\n# Fixed code\narr = np.array([1.0], dtype=np.float64)"},
    {"instruction": "Error: git error: pathspec 'main' did not match any file(s) known to git", "output": "What broke: Git cannot find a branch called 'main' in your repository.\nWhy it broke: The branch might be called 'master' instead of 'main', or the branch simply does not exist yet.\nHow to fix: Check what branches exist and use the correct name.\nExample fix:\ngit branch -a  # list all branches\ngit checkout master  # or whatever branch name exists"},
    {"instruction": "Error: IndentationError: unindent does not match any outer indentation level", "output": "What broke: Python found inconsistent indentation in your code, mixing tabs and spaces.\nWhy it broke: You used tabs for some lines and spaces for others. Python is very strict about consistent indentation.\nHow to fix: Use only spaces (4 spaces per level) throughout your entire file. Configure your editor to convert tabs to spaces.\nExample fix:\n# Use consistent 4-space indentation everywhere\ndef my_function():\n    if True:\n        print('hello')  # 8 spaces, consistent"},
    {"instruction": "Error: TypeError: object of type 'NoneType' has no len()", "output": "What broke: You called len() on a variable that is None instead of a list, string, or other sequence.\nWhy it broke: A function returned None instead of a collection, or a variable was never assigned a value.\nHow to fix: Check if the variable is None before calling len() on it.\nExample fix:\nresult = get_items()\nif result is not None:\n    print(len(result))\nelse:\n    print('No items found')"},
    {"instruction": "Error: ConnectionRefusedError: [Errno 111] Connection refused", "output": "What broke: Your code tried to connect to a server or port but nothing is listening there.\nWhy it broke: The server application is not running, it crashed, or you specified the wrong port number.\nHow to fix: Make sure the server is running before connecting to it, and verify the correct host and port.\nExample fix:\n# Check if your server is actually running first\n# Then connect with the correct port\nimport socket\ns = socket.socket()\ntry:\n    s.connect(('localhost', 8080))\nexcept ConnectionRefusedError:\n    print('Server is not running on port 8080')"},
    {"instruction": "Error: DeprecationWarning: an integer is required (got type float)", "output": "What broke: A function that requires an integer received a float value instead.\nWhy it broke: You passed a decimal number like 2.5 or a result of division to a function that only works with whole numbers.\nHow to fix: Convert the float to an integer using int() before passing it to the function.\nExample fix:\nindex = total / 2\n# Fix: convert to int\nindex = int(total / 2)\n# Or use integer division\nindex = total // 2"},
    {"instruction": "Error: RuntimeError: dictionary changed size during iteration", "output": "What broke: You tried to add or remove items from a dictionary while looping over it at the same time.\nWhy it broke: Python does not allow modifying a dictionary's structure while iterating over it because it can cause undefined behavior.\nHow to fix: Create a copy of the dictionary keys first, then iterate over the copy while modifying the original.\nExample fix:\nfor key in list(my_dict.keys()):  # iterate over a copy\n    if condition:\n        del my_dict[key]  # safe to modify original"},
    {"instruction": "Error: ValueError: shapes (3,) and (4,) not aligned for dot product", "output": "What broke: You tried to multiply two arrays or matrices but their shapes are incompatible for matrix multiplication.\nWhy it broke: Matrix multiplication requires the number of columns in the first matrix to equal the number of rows in the second. Your dimensions do not match.\nHow to fix: Check the shapes of both arrays and reshape them to be compatible.\nExample fix:\nimport numpy as np\na = np.array([1, 2, 3])  # shape (3,)\nb = np.array([1, 2, 3])  # shape (3,)\nresult = np.dot(a, b)  # works because both are (3,)"},
    {"instruction": "Error: SSL: CERTIFICATE_VERIFY_FAILED", "output": "What broke: Python could not verify the SSL certificate of the server you tried to connect to.\nWhy it broke: The certificate is expired, self-signed, or your system's certificate store is outdated.\nHow to fix: Update your certificates. As a temporary workaround you can disable verification but this is not recommended for production.\nExample fix:\n# Update certificates on Mac\n# /Applications/Python 3.x/Install Certificates.command\n# Temporary workaround only - not safe for production\nimport requests\nresponse = requests.get(url, verify=False)"},
    {"instruction": "Error: TypeError: list indices must be integers or slices, not str", "output": "What broke: You tried to access a list using a string key like a dictionary instead of an integer index.\nWhy it broke: Lists use numeric indexes starting at 0. You might have confused a list with a dictionary, or your data structure is different from what you expected.\nHow to fix: Use an integer index for lists, or check if you actually have a dictionary instead of a list.\nExample fix:\n# If it's a list\nmy_list = ['a', 'b', 'c']\nprint(my_list[0])  # use integer index\n# If it's a dict\nmy_dict = {'name': 'Alice'}\nprint(my_dict['name'])  # use string key"},
    {"instruction": "Error: EnvironmentError: CUDA driver version is insufficient", "output": "What broke: The CUDA driver installed on your system is too old for the version of PyTorch or TensorFlow you are using.\nWhy it broke: Deep learning libraries require specific minimum CUDA driver versions. Newer library versions need newer drivers.\nHow to fix: Update your NVIDIA GPU driver, or install an older version of PyTorch that matches your current driver.\nExample fix:\n# Check your CUDA version\nnvidia-smi\n# Install matching PyTorch version from pytorch.org\n# e.g. for CUDA 11.8:\npip install torch --index-url https://download.pytorch.org/whl/cu118"},
    {"instruction": "Error: KeyboardInterrupt", "output": "What broke: Nothing actually broke. You pressed Ctrl+C to manually stop the program while it was running.\nWhy it broke: This is intentional interruption, not a real error. It means the program was running (possibly stuck or taking too long) and you stopped it.\nHow to fix: This is usually fine. If you want to handle it gracefully in your own code you can catch it.\nExample fix:\ntry:\n    long_running_process()\nexcept KeyboardInterrupt:\n    print('Process stopped by user')\n    cleanup_resources()"},
]

print(f"Total training pairs: {len(data)}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print("Loaded on:", next(model.parameters()).device)

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

lora = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj"]
)

dataset = Dataset.from_list([
    {"text": f"<s>[INST] Explain this error to a beginner: {d['instruction']} [/INST] {d['output']} </s>"}
    for d in data
])

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./bug_explainer",
        num_train_epochs=8,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        max_length=512,
        fp16=True,
        bf16=False,
        logging_steps=10,
        dataloader_pin_memory=False,
    ),
    peft_config=lora,
)

trainer.train()
trainer.save_model("./bug_explainer")
print("Done!")

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="./bug_explainer",
    tokenizer=tokenizer,
    torch_dtype=torch.float16,
    device_map="auto"
)

tests = [
    "NameError: name 'x' is not defined",
    "TypeError: unsupported operand type(s) for +: 'int' and 'str'",
    "IndexError: list index out of range",
    "Cannot read properties of undefined (reading 'map')",
    "NullPointerException at line 42",
]

for t in tests:
    prompt = f"<s>[INST] Explain this error to a beginner: {t} [/INST]"
    result = pipe(
        prompt,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.3,
        repetition_penalty=1.4,
        eos_token_id=tokenizer.eos_token_id,
    )
    answer = result[0]["generated_text"].split("[/INST]")[-1].strip()
    print(f"Error: {t}")
    print(answer)
    print("-" * 60)

In [ ]:
from peft import PeftModel
import shutil

base = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    torch_dtype=torch.float16,
    device_map="auto"
)
merged = PeftModel.from_pretrained(base, "./bug_explainer")
merged = merged.merge_and_unload()
merged.save_pretrained("./bug_explainer_final", safe_serialization=True)
tokenizer.save_pretrained("./bug_explainer_final")
print("Saved!")

In [ ]:
shutil.make_archive("bug_explainer_final", "zip", "./bug_explainer_final")
from google.colab import files
files.download("bug_explainer_final.zip")